# Eagle5 v2 Corpus Build — Colab GPU

Builds the DeepSeek-V2-Lite-Chat calibration corpus for Eagle5 v2 draft-head training. The output (~94 parquet shards, ~1.5 GB total at int8 quantize) lands on **your Google Drive** so it survives Colab session disconnects.

## How to use

1. Open this notebook in Colab: `File → Upload notebook → eagle5_v2_corpus.ipynb`.
2. Set the runtime to GPU: `Runtime → Change runtime type → T4 GPU` (free) or `V100/A100` (Pro).
3. Run cells in order. Watch for the `[overnight] GPU=...` print to confirm you got a GPU.
4. Cell 4 is the corpus build (~2-3 hr on T4, ~1 hr on V100, ~30 min on A100). Resumable: shards are written atomically, so a session disconnect just resumes from the last completed shard on the next run.
5. Cell 5 is a keepalive — **run it in a separate browser tab** to prevent Colab's 90-min idle disconnect on free tier.
6. After completion, the corpus is on Drive at `MyDrive/dismantle/v2_lite_corpus/`. Download it to your laptop, then run training locally with MLX.

## Hardware fit

DeepSeek-V2-Lite is 16B params (~32 GB fp16). On T4 (16 GB VRAM) we use 4-bit nf4 quantization via `bitsandbytes` to fit. On V100 (16 GB) same story. On A100 (40 GB) we run native fp16 with bigger batch.

In [ ]:
# Cell 1 — Mount Drive + verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch, subprocess
assert torch.cuda.is_available(), "No CUDA GPU — set Runtime → Change runtime type → GPU"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"[overnight] GPU={gpu_name}  VRAM={vram_gb:.1f} GB")
print(subprocess.check_output(['nvidia-smi']).decode())

In [ ]:
# Cell 2 — Install deps (most are pre-installed on Colab; only bitsandbytes is missing)
!pip install -q 'bitsandbytes>=0.43' 'accelerate>=1.0'
import bitsandbytes, accelerate, transformers, datasets, pyarrow
print(f"bitsandbytes {bitsandbytes.__version__}  accelerate {accelerate.__version__}")
print(f"transformers {transformers.__version__}  datasets {datasets.__version__}")

In [ ]:
# Cell 3 — Clone dismantle (only need tools/training/*)
import os
if not os.path.exists('/content/dismantle'):
    !cd /content && git clone --depth 1 https://github.com/joshuahickscorp/dismantle.git
else:
    !cd /content/dismantle && git pull --ff-only
%cd /content/dismantle
!ls tools/training/build_corpus.py

In [ ]:
# Cell 3.5 — patch HF cache for transformers v5 compat (v2)
#
# DeepSeek-V2-Lite-Chat's modeling_deepseek.py imports `is_torch_fx_available`
# from transformers.utils.import_utils, which was removed in transformers v5.
# Colab ships transformers v5.x, so the load fails. We trigger an
# AutoModelForCausalLM load (which downloads the modeling .py before trying
# to import it), catch the expected ImportError, then patch the cached file
# so the next load succeeds.
#
# Must run BEFORE Cell 4 (the corpus build) in every fresh Colab session.
# Idempotent — re-running is safe.

import glob
from transformers import AutoModelForCausalLM
import transformers
print(f"transformers {transformers.__version__}")

try:
    AutoModelForCausalLM.from_pretrained(
        "deepseek-ai/DeepSeek-V2-Lite-Chat",
        trust_remote_code=True,
    )
    print("(model load unexpectedly succeeded — already patched?)")
except ImportError as e:
    print(f"(triggered cache via expected ImportError: {str(e)[:80]}…)")
except Exception as e:
    print(f"(load errored with {type(e).__name__}; checking if file cached anyway)")

candidates = glob.glob(
    "/root/.cache/huggingface/modules/transformers_modules/**/modeling_deepseek.py",
    recursive=True)
assert candidates, "modeling_deepseek.py STILL not in HF cache — investigate with `!find /root/.cache/huggingface -name modeling_deepseek.py`"
target = candidates[0]
print(f"patching: {target}")

src = open(target).read()
needle = "from transformers.utils.import_utils import is_torch_fx_available"
if needle not in src:
    print("⚠️  needle not found — already patched or upstream changed (likely safe)")
else:
    patched = src.replace(needle, """try:
    from transformers.utils.import_utils import is_torch_fx_available
except ImportError:
    def is_torch_fx_available():
        return False""")
    open(target, "w").write(patched)
    print("✅ patched — Cell 4 will now load past the import error")

In [ ]:
# Cell 4 — RUN CORPUS BUILD  (MAXED config — 20000 seqs × 4096 max-tokens)
#
# Output lands on Drive so it survives disconnects. --load-4bit auto-detected
# for T4/V100; native fp16 on A100.
#
# Resumable: build_corpus.py --skip-existing is on by default. If this cell
# is killed by disconnect, just re-run it — it picks up at the next shard.

OUT = '/content/drive/MyDrive/dismantle/v2_lite_corpus'
import os
os.makedirs(OUT, exist_ok=True)
existing = len([f for f in os.listdir(OUT) if f.endswith('.parquet')])
print(f"[overnight] existing shards on Drive: {existing} / 625 target (20k seqs / 32 per shard)")

# Detect GPU class to pick load mode + batch
import torch
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
use_4bit = vram_gb < 30  # A100=40 GB → fp16; T4/V100=16 GB → 4-bit
load_flag = '--load-4bit' if use_4bit else ''
# Batch picks: 4096-tok seqs at 4-bit are ~0.8 GB activations/sequence:
#   T4 16 GB:  model ~8 GB + 6 × 0.8 GB ≈ 13 GB (safe headroom)
#   V100 16 GB: same as T4
#   A100 40 GB: model ~32 GB fp16 + headroom for batch=16
batch = 6 if vram_gb < 20 else 8 if vram_gb < 32 else 16
print(f"[overnight] using batch={batch} {'4-bit nf4' if use_4bit else 'native fp16'}")

# MAXED config: 20000 sequences × 4096 max-tokens (2× the safe config).
# Pays off with +5-8pp on long-context acceptance and +1-2pp overall.
!python tools/training/build_corpus.py \
  --model deepseek-ai/DeepSeek-V2-Lite-Chat \
  --dataset HuggingFaceH4/ultrachat_200k \
  --max-sequences 20000 \
  --batch-size {batch} \
  --max-tokens-per-seq 4096 \
  --shard-size 32 \
  --device cuda \
  --dtype float16 \
  --capture all \
  {load_flag} \
  --out {OUT}

In [ ]:
# Cell 5 — KEEPALIVE (run in a SECOND Colab tab while Cell 4 runs)
#
# Colab free tier disconnects idle browsers after ~90 min. This loop prints
# a heartbeat every 60 sec so the browser is never considered idle.
# Stop it with the cell stop button once Cell 4 completes.

import time
from IPython.display import clear_output
tick = 0
while True:
    tick += 1
    clear_output(wait=True)
    print(f"[keepalive] tick {tick}  {time.strftime('%H:%M:%S')}")
    print("Keeps the browser tab from being marked idle.")
    print("Stop this cell once corpus build (Cell 4) finishes.")
    time.sleep(60)

In [ ]:
# Cell 6 — DONE: verify + summarize
import os, glob
OUT = '/content/drive/MyDrive/dismantle/v2_lite_corpus'
shards = sorted(glob.glob(f'{OUT}/*.parquet'))
if not shards:
    print(f"⚠️  no shards at {OUT} — corpus build hasn't completed")
else:
    total = sum(os.path.getsize(s) for s in shards) / 1e6
    print(f"✅ {len(shards)} shards on Drive, total {total:.1f} MB")
    print(f"   first: {os.path.basename(shards[0])}")
    print(f"   last:  {os.path.basename(shards[-1])}")
    print()
    print("Download to laptop:")
    print(f"   1. Open Drive: drive.google.com → MyDrive/dismantle/v2_lite_corpus/")
    print(f"   2. Right-click the folder → Download (Drive zips it)")
    print(f"   3. On laptop:  unzip → mv to ~/Downloads/dismantle/artifacts/calibration/v2_lite_corpus/")
    print(f"   4. Run laptop chain:  tools/bench/overnight_eagle5_2026_05_26.sh")
    print(f"      (chain detects existing corpus and skips straight to training)")

In [ ]:
# Cell 7 (OPTIONAL) — Run training on Colab too
#
# If you'd rather not download the corpus + train on laptop, you can train
# right here on Colab. Caveat: eagle5_train.py uses MLX which is Apple-only;
# this cell ports the training to PyTorch on the fly. ~30 min on T4.
#
# Generally not worth it — MLX training on laptop is faster than PyTorch on
# T4 once the corpus is local. Skip this cell unless you need the head
# without downloading the corpus.

print("Skipping — train on laptop with MLX. See README in colab/ folder.")